# load in spectra

In [3]:
import os
from matchms.importing import load_from_mgf
from tqdm import tqdm


save_directory = "../data/ms2deepscore_model/training_and_validation_split/"
pos_val_spectra = list(tqdm(load_from_mgf(os.path.join(save_directory, "positive_validation_spectra.mgf"))))
neg_val_spectra = list(tqdm(load_from_mgf(os.path.join(save_directory, "negative_validation_spectra.mgf"))))
pos_train_spectra = list(tqdm(load_from_mgf(os.path.join(save_directory, "positive_training_spectra.mgf"))))
neg_train_spectra = list(tqdm(load_from_mgf(os.path.join(save_directory, "negative_training_spectra.mgf"))))
pos_test_spectra = list(tqdm(load_from_mgf(os.path.join(save_directory, "positive_testing_spectra.mgf"))))
neg_test_spectra = list(tqdm(load_from_mgf(os.path.join(save_directory, "negative_testing_spectra.mgf"))))


31453it [00:14, 2140.47it/s]
7080it [00:03, 1992.44it/s]
459610it [03:37, 2114.18it/s]
131240it [01:06, 1967.67it/s]


### Save as pickled files for quicker reloads

In [4]:
import os


pickled_intermediates_data_folder = "../data/pickled_intermediates"
os.path.isdir(pickled_intermediates_data_folder)

True

In [6]:
import pickle


with open(os.path.join(pickled_intermediates_data_folder, "neg_val_spectra.pickle"), "wb") as handle:
    pickle.dump(neg_val_spectra, handle, protocol=pickle.HIGHEST_PROTOCOL)
with open(os.path.join(pickled_intermediates_data_folder, "neg_train_spectra.pickle"), "wb") as handle:
    pickle.dump(neg_train_spectra, handle, protocol=pickle.HIGHEST_PROTOCOL)
with open(os.path.join(pickled_intermediates_data_folder, "pos_val_spectra.pickle"), "wb") as handle:
    pickle.dump(pos_val_spectra, handle, protocol=pickle.HIGHEST_PROTOCOL)
with open(os.path.join(pickled_intermediates_data_folder, "pos_train_spectra.pickle"), "wb") as handle:
    pickle.dump(pos_train_spectra, handle, protocol=pickle.HIGHEST_PROTOCOL)
with open(os.path.join(pickled_intermediates_data_folder, "pos_test_spectra.pickle"), "wb") as handle:
    pickle.dump(pos_test_spectra, handle, protocol=pickle.HIGHEST_PROTOCOL)
with open(os.path.join(pickled_intermediates_data_folder, "neg_test_spectra.pickle"), "wb") as handle:
    pickle.dump(neg_test_spectra, handle, protocol=pickle.HIGHEST_PROTOCOL)

# Create a simple MS2Deepscore ranker

In [ ]:
from ms2deepscore.models import load_model


ms2deepscore_model = load_model(
    "../data/ms2deepscore_model/trained_models/"
    "both_mode_ionmode_precursor_mz_2000_layers_500_embedding_2025_02_26_18_42_25/ms2deepscore_model.pt")


In [ ]:
import sys


sys.path.append("../../ms_chemical_space_explorer")

In [ ]:
from ms_chemical_space_explorer.SpectrumDataSet import SpectraWithMS2DeepScoreEmbeddings


library_spectra = SpectraWithMS2DeepScoreEmbeddings(neg_train_spectra + pos_train_spectra, ms2deepscore_model)

In [ ]:
import pickle


with open(os.path.join(pickled_intermediates_data_folder, "neg_pos_library_embeddings.pickle"), "wb") as handle:
    pickle.dump(library_spectra.embeddings, handle, protocol=pickle.HIGHEST_PROTOCOL)

In [ ]:
import pickle


with open(os.path.join(pickled_intermediates_data_folder, "neg_pos_library_with_embeddings.pickle"), "wb") as handle:
    pickle.dump(library_spectra, handle, protocol=pickle.HIGHEST_PROTOCOL)

In [ ]:
val_spectra =  SpectraWithMS2DeepScoreEmbeddings(neg_val_spectra + pos_val_spectra, ms2deepscore_model)

In [ ]:
import pickle


with open(os.path.join(pickled_intermediates_data_folder,
                       "neg_pos_val_spectra_with_embeddings.pickle"), "wb") as handle:
    pickle.dump(val_spectra, handle, protocol=pickle.HIGHEST_PROTOCOL)

# Reload intermediates

In [4]:
import sys


sys.path.append("../../ms_chemical_space_explorer")

In [5]:
import os
import pickle


pickled_intermediates_data_folder = "../data/pickled_intermediates"
with open(os.path.join(pickled_intermediates_data_folder, "neg_pos_library_with_embeddings.pickle"), "rb") as file:
    library_spectra = pickle.load(file)
with open(os.path.join(pickled_intermediates_data_folder, "neg_pos_val_spectra_with_embeddings.pickle"), "rb") as file:
    val_spectra = pickle.load(file)

# Initialize a method evaluator

In [6]:
from ms_chemical_space_explorer.benchmarking.EvaluateMethods import EvaluateMethods


method_evaluator = EvaluateMethods(library_spectra, val_spectra)
method_evaluator.training_spectrum_set.progress_bars = False
method_evaluator.validation_spectrum_set.progress_bars = False

# Test basic MS2DeepScore

In [21]:
from ms_chemical_space_explorer.methods.predict_highest_ms2deepscore import predict_highest_ms2deepscore


result_analogue = method_evaluator.benchmark_analogue_search(predict_highest_ms2deepscore)

Predicting highest ms2deepscore per batch of 500 embeddings: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 78/78 [26:20<00:00, 20.26s/it]
Calculating analogue accuracy per inchikey: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 2015/2015 [00:01<00:00, 1190.75it/s]


In [22]:
print(result_analogue)

0.3848227909492443


In [23]:
from ms_chemical_space_explorer.methods.predict_highest_ms2deepscore import predict_highest_ms2deepscore


result_positive = method_evaluator.benchmark_exact_matching_within_ionmode(predict_highest_ms2deepscore, "positive")

Splitting spectra per inchikey: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1837/1837 [00:00<00:00, 53337.60it/s]
Predicting highest ms2deepscore per batch of 500 embeddings: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 31/31 [10:21<00:00, 20.03s/it]
Predicting highest ms2deepscore per batch of 500 embeddings: 100%|████████████████████████████████████████████████████████████████████████████████████████

In [24]:
print(result_positive)

0.5169110149857257


In [25]:
from ms_chemical_space_explorer.methods.predict_highest_ms2deepscore import predict_highest_ms2deepscore


result_neg = method_evaluator.benchmark_exact_matching_within_ionmode(predict_highest_ms2deepscore, "negative")

Splitting spectra per inchikey: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 924/924 [00:00<00:00, 293846.15it/s]
Predicting highest ms2deepscore per batch of 500 embeddings: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 7/7 [02:08<00:00, 18.29s/it]
Predicting highest ms2deepscore per batch of 500 embeddings: 100%|████████████████████████████████████████████████████████████████████████████████████████

In [26]:
result_neg


0.5448453174876924

In [32]:
result_across_ionmodes = method_evaluator.exact_matches_across_ionization_modes(predict_highest_ms2deepscore)

Splitting spectra per inchikey across ionmodes: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 2015/2015 [00:00<00:00, 9551.01it/s]
Predicting highest ms2deepscore per batch of 500 embeddings: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 35/35 [11:40<00:00, 20.02s/it]
Predicting highest ms2deepscore per batch of 500 embeddings: 100%|████████████████████████████████████████████████████████████████████████████████████████

In [33]:
result_across_ionmodes

0.0006188308440879157

# Test best possible results

In [4]:
from ms_chemical_space_explorer.methods.predict_best_possible_match import predict_best_possible_match


In [6]:
result_analogue_best = method_evaluator.benchmark_analogue_search(predict_highest_ms2deepscore)

Calculating tanimoto scores to determine best possible match


Calculating analogue accuracy per inchikey: 100%|██████████████████████████████████████████| 2015/2015 [00:01<00:00, 1578.04it/s]


In [7]:
result_analogue_best

0.7753653963061775

In [27]:
result_neg_best = method_evaluator.benchmark_exact_matching_within_ionmode(predict_best_possible_match, "negative")

Splitting spectra per inchikey: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 924/924 [00:00<00:00, 310739.01it/s]


Calculating tanimoto scores to determine best possible match
Calculating tanimoto scores to determine best possible match


Calculating exact match accuracy per inchikey: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 868/868 [00:00<00:00, 928738.74it/s]


In [29]:
result_neg_best

1.0

In [28]:
result_across_ionmodes_best = method_evaluator.exact_matches_across_ionization_modes(predict_best_possible_match)
result_across_ionmodes_best

Splitting spectra per inchikey across ionmodes: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 2015/2015 [00:00<00:00, 13798.67it/s]


Calculating tanimoto scores to determine best possible match
Calculating tanimoto scores to determine best possible match


Calculating exact match accuracy per inchikey: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 746/746 [00:00<00:00, 404727.82it/s]


1.0

In [5]:
result_across_ionmodes_best = method_evaluator.exact_matches_across_ionization_modes(predict_best_possible_match)

Splitting spectra per inchikey across ionmodes: 100%|█████████████████████████████████████| 2015/2015 [00:00<00:00, 10356.24it/s]


Calculating tanimoto scores to determine best possible match
Calculating tanimoto scores to determine best possible match


Calculating exact match accuracy per inchikey: 100%|███████████████████████████████████████| 746/746 [00:00<00:00, 315164.26it/s]


In [8]:
result_across_ionmodes_best

1.0

# Test cosine score

In [4]:
from ms_chemical_space_explorer.methods.predict_highest_cosine import predict_highest_cosine

In [ ]:
result_analogue_cosine = method_evaluator.benchmark_analogue_search(predict_highest_cosine)

In [ ]:
result_analogue_cosine

In [24]:
result_neg = method_evaluator.benchmark_exact_matching_within_ionmode(predict_highest_cosine, "negative")

Splitting spectra per inchikey: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 924/924 [00:00<00:00, 260575.33it/s]


created scores
Filtered on precursor
Calculated cosine
('PrecursorMzMatch', 'CosineGreedy_score', 'CosineGreedy_matches')
created scores
Filtered on precursor
Calculated cosine
('PrecursorMzMatch', 'CosineGreedy_score', 'CosineGreedy_matches')


Calculating exact match accuracy per inchikey: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 868/868 [00:00<00:00, 201408.27it/s]


In [25]:
result_neg

0.5772003486330072

In [ ]:
result_positive = method_evaluator.benchmark_exact_matching_within_ionmode(predict_highest_cosine, "positive")

In [ ]:
result_positive

In [ ]:
result_across_ionmodes = method_evaluator.exact_matches_across_ionization_modes(predict_highest_cosine)

In [ ]:
result_across_ionmodes

# Test predictions with ISF

In [ ]:
from ms_chemical_space_explorer.methods.predict_with_integrated_similarity_flow import (
    predict_with_integrated_similarity_flow,
)


result_analogue_isf = method_evaluator.benchmark_analogue_search(predict_with_integrated_similarity_flow)
print(result_analogue_isf)
result_neg_isf = method_evaluator.benchmark_exact_matching_within_ionmode(
    predict_with_integrated_similarity_flow, "negative")
print(result_neg_isf)
result_positive_isf = method_evaluator.benchmark_exact_matching_within_ionmode(
    predict_with_integrated_similarity_flow, "positive")
print(result_positive_isf)
result_across_ionmodes_isf = method_evaluator.exact_matches_across_ionization_modes(
    predict_with_integrated_similarity_flow)
print(result_across_ionmodes_isf)

In [ ]:
result_analogue_isf

In [1]:
print("hello")

hello
